## Exercise 6: Shared utility functions, data catalogs

Skills: 
* Import shared utils
* Data catalog
* Use functions to repeat certain data cleaning steps

References: 
* https://docs.calitp.org/data-infra/analytics_new_analysts/02-data-analysis-intermediate.html
* https://docs.calitp.org/data-infra/analytics_tools/python_libraries.html
* https://docs.calitp.org/data-infra/analytics_tools/data_catalogs.html

In [1]:
import geopandas as gpd
import intake
import pandas as pd

from calitp_data_analysis import geography_utils, utils
from calitp_data_analysis import styleguide

# Hint: if this doesn't import: refer to docs for correctly import
# cd into _shared_utils folder, run the make setup_env command
from shared_utils import portfolio_utils

## Create a data catalog

* Include one geospatial data source and one tabular (they should be related...your analysis depends on combining them)
* Import your datasets using the catalog method

In [31]:
catalog = intake.open_catalog("sample_catalog.yml")

In [32]:
counties = catalog.caltrans_open_data.ca_county_boundaries.read()
counties.head()

,OBJECTID,COUNTYFP10,GEOID10,NAME10,ALAND10,AWATER10,INTPTLAT10,INTPTLON10,CO_CODE,DISTRICT,SHAPE_Length,SHAPE_Area,geometry
0,1,059,06059,Orange,2.047561e+09,4.079168e+08,+33.6756872,-117.7772068,ORA,12,2.978872,0.200321,"MULTIPOLYGON (((-117.91576 33.94693, -117.9156..."
1,2,103,06103,Tehama,7.639710e+09,3.227606e+07,+40.1261561,-122.2322757,TEH,2,5.152453,0.810614,"MULTIPOLYGON (((-121.68552 40.45308, -121.6854..."
2,3,011,06011,Colusa,2.980379e+09,1.458102e+07,+39.1777385,-122.2375629,COL,3,3.212895,0.312200,"MULTIPOLYGON (((-122.10654 39.41442, -122.1027..."
3,4,083,06083,Santa Barbara,7.083838e+09,2.729829e+09,+34.5373784,-120.0384850,SB,5,4.252901,0.651046,"MULTIPOLYGON (((-120.07950 35.11300, -120.0793..."
4,5,051,06051,Mono,7.896827e+09,2.146955e+08,+37.9158363,-118.8751668,MNO,9,5.686027,0.830964,"MULTIPOLYGON (((-117.83340 37.46494, -117.8348..."


In [33]:
federal_jobs = catalog.lehd_federal_jobs_by_tract.read()
federal_jobs.head()

,st,stname,cty,ctyname,trct,trctname,year,c000,ca01,ca02,...,cr05,cr07,ct01,ct02,cd01,cd02,cd03,cd04,cs01,cs02
0,1,Alabama,1001,"Autauga County, AL",1001020100,"201 (Autauga, AL)",2011,3,0,2,...,0,0,3,0,0,3,0,0,0,3
1,1,Alabama,1001,"Autauga County, AL",1001020100,"201 (Autauga, AL)",2012,1,0,1,...,0,0,1,0,0,0,0,1,1,0
2,1,Alabama,1001,"Autauga County, AL",1001020100,"201 (Autauga, AL)",2013,2,0,1,...,0,0,2,0,0,2,0,0,2,0
3,1,Alabama,1001,"Autauga County, AL",1001020200,"202 (Autauga, AL)",2011,2,0,0,...,0,0,2,0,0,0,1,1,2,0
4,1,Alabama,1001,"Autauga County, AL",1001020200,"202 (Autauga, AL)",2012,4,0,3,...,0,0,4,0,0,0,3,1,3,1


## Combine datasets
* Do a merge or spatial join to combine the geospatial and tabular data
* Create a new column of a summary statistic to visualize
* Rely on `calitp_data_analysis` or `shared_utils` to do at least one operation (aggregation, re-projecting to a different CRS, exporting geoparquet, etc)
   * aggregation: `portfolio_utils.aggregate_by_geography`
   * exporting geoparquet: `calitp_data_analysis.utils.geoparquet_gcs_export`

In [61]:
federal_jobs_ca = federal_jobs.loc[federal_jobs["st"] == 6].copy()
federal_jobs_ca["county_three_digit"] = federal_jobs_ca["cty"].astype(str).str.slice(start=1)
federal_jobs_totals = federal_jobs_ca.drop(["st", "stname", "cty", "ctyname", "trct", "trctname", "year"], axis=1).groupby("county_three_digit").sum()
counties_with_federal_jobs = counties.merge(
    federal_jobs_totals,
    how="left",
    left_on="COUNTYFP10",
    right_index=True,
    validate="one_to_one"
).set_index("COUNTYFP10")
counties_with_federal_jobs

,OBJECTID,GEOID10,NAME10,ALAND10,AWATER10,INTPTLAT10,INTPTLON10,CO_CODE,DISTRICT,SHAPE_Length,...,cr05,cr07,ct01,ct02,cd01,cd02,cd03,cd04,cs01,cs02
COUNTYFP10,,,,,,,,,,,,,,,,,,,,,
059,1,06059,Orange,2.047561e+09,4.079168e+08,+33.6756872,-117.7772068,ORA,12,2.978872,...,280,2793,27903,6039,407,7062,7664,16004,16893,17049
103,2,06103,Tehama,7.639710e+09,3.227606e+07,+40.1261561,-122.2322757,TEH,2,5.152453,...,3,40,1118,63,2,251,264,555,753,428
011,3,06011,Colusa,2.980379e+09,1.458102e+07,+39.1777385,-122.2375629,COL,3,3.212895,...,60,188,4365,350,14,206,1417,2581,2170,2545
083,4,06083,Santa Barbara,7.083838e+09,2.729829e+09,+34.5373784,-120.0384850,SB,5,4.252901,...,40,744,7270,2068,27,3162,2455,2840,6225,3113
051,5,06051,Mono,7.896827e+09,2.146955e+08,+37.9158363,-118.8751668,MNO,9,5.686027,...,2,10,195,16,0,57,50,84,137,74
053,6,06053,Monterey,8.496703e+09,1.270714e+09,+36.2401070,-121.3155732,MON,5,6.542538,...,99,995,3849,2548,135,1739,2216,1885,2118,4279
061,7,06061,Placer,3.644136e+09,2.472054e+08,+39.0620324,-120.7227181,PLA,3,4.373307,...,17,280,2629,654,55,960,923,947,2032,1251
005,8,06005,Amador,1.539963e+09,2.945658e+07,+38.4435501,-120.6538563,AMA,10,2.909737,...,3,156,709,253,11,395,216,279,528,434
009,9,06009,Calaveras,2.641820e+09,4.381042e+07,+38.1878437,-120.5551154,CAL,10,2.963054,...,0,17,202,19,0,67,75,55,155,66


## Use functions to do parameterized visualizations
* Use a function to create your chart
* Within the function, the colors should use the Cal-ITP theme that is available in `styleguide`
* Within the function, there should be at least 1 parameter that changes (ex: chart title reflects the correct county, legend title reflects the correct county, etc)
* Produce 3 charts, using your function each time, and have the function correctly insert the parameters 

In [94]:
import altair as alt
from branca.colormap import LinearColormap

RED_GREEN_RANGE = ["#FF2020", "#EFF520", "#20FF20"]

def job_types_by_county(df: pd.DataFrame, three_digit_county_fips: str):
    jobs_in_county = df.loc[[three_digit_county_fips], :].copy()
    
    jobs_in_county_long = jobs_in_county.melt(
        id_vars=["OBJECTID", "GEOID10", "NAME10", "ALAND10", "AWATER10", "INTPTLAT10", "INTPTLON10", "CO_CODE", "DISTRICT", "SHAPE_Length", "SHAPE_Area", "geometry"],
        var_name="Category Code",
        value_name="# Jobs"
    )
    county_name = str.capitalize(jobs_in_county.at[three_digit_county_fips, "NAME10"])
    chart = alt.Chart(
        jobs_in_county_long, title=f"Federal Jobs in {county_name} by Categories"
    ).mark_bar(
        size=10
    ).encode(
        x=alt.X("Category Code"),
        y=alt.Y("# Jobs"),
        color=alt.Color("# Jobs").scale(domain=[jobs_in_county_long["# Jobs"].min(), jobs_in_county_long["# Jobs"].max()], range=RED_GREEN_RANGE)
    )
    return chart 

In [95]:
for county_fips in ["087", "037", "073"]:
    job_types_by_county(counties_with_federal_jobs, county_fips)

alt.Chart(...)